# ocean: line

ocean transport data cross defined lines

```{dropdown}  **Branding suffix of datasets**
Datasets with the folloing branding suffix are shown in this page: \
[time]-[vertical]-[horizontal]-[domain]
```
* tavg-u-ht-sea
    - tavg: time average
    - u: ocean surface
    - ht: defined lines
    - sea: ocean domain

In [ ]:
## Import libraries
import glob
import logging
import os

import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import HTML, display
from pathlib import Path
from pyprojroot import here
from xarray.coders import CFDatetimeCoder

from cmip7validate.utils import read_compound_names, read_variables

logging.basicConfig(
    filename="execute.log",
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    force=True,
)
log = logging.getLogger("ocean_line")
log.info("Starting ocean_line.ipynb")

# Get project root
root_path = Path(here())

In [ ]:
# load parameters
import yaml

with open(root_path.joinpath('params.yml'), 'r') as f:
    param = yaml.safe_load(f)
    cmorout = param['cmorout']
    source_id = param['source_id']
    experiment_id = param['experiment_id']
    variant_label = param['variant_label']
    version = param['version']

In [ ]:
# data path
data_path = Path(cmorout).joinpath(source_id, experiment_id, version)

# load methods for plotting and set defaults
methods =read_variables(root_path.joinpath('assets/methods.txt'))
#print(methods.keys())

---
```{dropdown}  **List of datasets:** 
List of compound names of datasets that needs to be cmorized. \
Those datasets which are not cmorized have no link.
```

In [ ]:
# load compound names
grid_label = 'g???'
coords = ('tavg-u-ht-sea')
cnames = read_compound_names(root_path.joinpath('assets/variables.nml'))
# examples of compound names:
# cnames = ['ocean.hfacrossline.tavg-u-ht-sea.mon.glb', 'ocean.mfo.tavg-u-ht-sea.mon.glb', 'ocean.sfacrossline.tavg-u-ht-sea.mon.glb']
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]
    
    if realm != 'ocean' or coord not in coords:
        continue
    else:
        file_pattern = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}_*.nc"
        if not sorted(data_path.glob(file_pattern)):
            print(cname)
            continue
        else:
            display(HTML(f'<a href="#{cname}">{cname}</a>'))

---
```{dropdown}  **Dataset validated:**
Each validated dataset has some text information and some plots (2D and/or 1D)
```

In [ ]:

# loop through compound names and plot
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]

    if realm != 'ocean' or coord not in coords:
        continue

    file_pattern = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}_*.nc"
    data_files = sorted(data_path.glob(file_pattern))
    if data_files:
        log.info(f"{cname}")
        with xr.open_mfdataset(data_files, decode_times=CFDatetimeCoder(use_cftime=True)) as ds:
            if var in ds:
                data = ds[var]
            else:
                continue
    else:
        continue

    display(HTML(f'<div id="{cname}"></div>'))
    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data.long_name} ({data.units})')
    print(f'original_name: {data.attrs["original_name"]} -> {var}')
    if 'history' in data.attrs:
        print(f'history: {data.attrs["history"]}')
    if 'comment' in data.attrs:
        print(f'comment: {data.attrs["comment"]}')    
    nlines = data.line.size
    sector = ds['sector'].data
    fig, axs = plt.subplots((nlines+1)//2, 2, figsize=(12, 16), dpi=96, sharex=True, sharey=False)
    ax = axs.flatten()
    for i in range(nlines):
        data.isel(line=i).plot(ax=ax[i])
        data.isel(line=i).plot(ax=ax[i])
        ax[i].set_title(sector[i].compute().astype(str).strip())
        ax[i].set_ylabel(data.units)
        ax[i].set_xlabel('')
        ax[i].grid()

    # Turn off x‑axis ticks for all subplots
    for ax in axs.flatten():
        ax.xaxis.set_tick_params(labelbottom=False, bottom=False)

    # Turn on x‑axis ticks only for the bottom row (last two axes)
    for ax in axs[-1,:]:
        ax.xaxis.set_tick_params(labelbottom=True, bottom=True)
        ax.set_xlabel('Time')

    fig.suptitle(f"{data.long_name}", fontsize=16)

    plt.tight_layout()
    plt.show()

    del data
    del ax, axs, fig